Write the data one row at a time by using CSV and "append" file writing mode, as demonstrated below.

In your real use case, you'd want to be making an API call, getting the data and then *writing the data to the CSV before* moving on to the next API call. 

You want to avoid making *all* of the API calls and collecting *all* of the data at once, because (1) that risks losing everything if the program crashes after running for a long time and (2) it will potentially strain your machine's memory.

Lastly you're giving yourself the ability to structure your code in a way that checks for the last stored record and can resume the scrape/API calling from the point that you things crashed (in the event of a bug).

In [28]:
import pandas as pd
import requests
import time
import re

In [29]:
# Setup
API_KEY = "yMgwnlBsql8axIBF9rnaf697f5tLlc9S2KYyO2P9"
BASE_URL = "https://commonchemistry.cas.org/api"
data_path = "data/CDPH Search results - 1_12_2026, 3_34_15 PM.csv"
data = pd.read_csv(data_path, low_memory = False)

# Function to extract CAS numbers that are already in the ingredient name
def get_cas_from_text(text):
    if not text.strip():
        return ""
    cas_numbers = re.findall(r'\b\d{2,7}-\d{2}-\d\b', str(text))
    return cas_numbers if cas_numbers else ""

# Function to clean ingredient name (remove CAS numbers)
def clean_name(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r'\d{2,7}-\d{2}-\d', '', text)  # Remove CAS
    text = re.sub(r'/\s*\d+[-\d/\s]*', '', text)   # Remove extra numbers
    return text.strip()


def get_official_name(unique_cas):
    if not unique_cas or unique_cas == "":
        return ""
    url = f"{BASE_URL}/detail"
    headers = {"X-Api-Key": API_KEY}
    params = {"cas_rn": unique_cas}
    try:
        response = requests.get(url, headers=headers, params=params)
        if response.status_code == 200:
            return response.json().get('name', 'NOT FOUND')
        else:
            return "NOT FOUND"
    except:
        return "ERROR"

In [37]:
# Setup
API_KEY = "yMgwnlBsql8axIBF9rnaf697f5tLlc9S2KYyO2P9"
BASE_URL = "https://commonchemistry.cas.org/api"
data_path = "data/CDPH Search results - 1_12_2026, 3_34_15 PM.csv"
data = pd.read_csv(data_path, low_memory = False)

# Function to extract CAS numbers that are already in the ingredient name
def get_cas_from_text(text):
    if not text.strip():
        return ""
    cas_numbers = re.findall(r'\b\d{2,7}-\d{2}-\d\b', str(text))
    return cas_numbers if cas_numbers else ""

# Function to clean ingredient name (remove CAS numbers)
def clean_name(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r'\d{2,7}-\d{2}-\d', '', text)  # Remove CAS
    text = re.sub(r'/\s*\d+[-\d/\s]*', '', text)   # Remove extra numbers
    text = re.sub(r'\s*/\s*', ' ', text)              # Replace / with spaces
    text = re.sub(r'\s+', ' ', text) 
    return text.strip()


def get_official_name(unique_cas):
    if not unique_cas or unique_cas == "":
        return ""
    url = f"{BASE_URL}/detail"
    headers = {"X-Api-Key": API_KEY}
    params = {"cas_rn": unique_cas}
    try:
        response = requests.get(url, headers=headers, params=params)
        if response.status_code == 200:
            return response.json().get('name', 'NOT FOUND')
        else:
            return "NOT FOUND"
    except:
        return "ERROR"

In [38]:
#Find CAS numbers
unique_cas = data["Ingredient Name"].unique()
#pd.DataFrame(unique_cas, columns = ["Ingredient Name"]).to_csv("ingredients.csv", index = False)

In [39]:
errors = []
cas_metadata = {}
for ingredient in unique_cas:
        cas_metadata[ingredient] = {}

In [40]:
cas_metadata

{'Titanium dioxide (CI 77891) 13463-67-7 / 1317-70-0 / 1317-80-2 / 98084-96-9': {},
 'CITRONELLOL ': {},
 'LINALOOL ': {},
 'HEXYL CINNAMAL ': {},
 'Limonene (1-methyl-4-prop-1-en-2-yl-cyclohexene; dl-limonene (racemic); Dipentene; (R)-p-mentha-1,8-diene; (d-limonene); (S)-p-mentha-1,8-diene; (l-limonene)) 5989-27-5 / 138-86-3 / 7705-14-8 / 5989-54-8': {},
 'Lilial ': {},
 'Aloe vera, non-decolorized (unfiltered) whole leaf extract ': {},
 '1,6-Octadien-3-ol, 3,7-dimethyl- (LINALOOL) 78-70-6': {},
 'CITRAL ': {},
 'GERANIOL ': {},
 'Citronellol /+- 3,7-dimethyloct-6-en-1-ol (CITRONELLOL; (3R)-3,7-dimethyloct-6-en-1-ol; (3S)-3,7-dimethyloct-6-en-1-ol) 106-22-9 / 26489-01-0 / 1117-61-9 / 7540-51-4': {},
 'Benzyl benzoate 120-51-4': {},
 'BUTYLPHENYL METHYLPROPIONAL ': {},
 '2-Benzylideneoctanal (HEXYL CINNAMAL) 101-86-0': {},
 '2-Phenoxyethyl isobutyrate 103-60-6': {},
 '1,3,4,6,7,8-Hexahydro-4,6,6,7,8,8-hexamethylcyclopenta-?-2-benzopyran (Hexamethylindanopyran) 1222-05-5': {},
 'Isoeug

In [41]:
# GO THROUGH DICTIONARY and apply ur functions
for ingredient, empty_dict in cas_metadata.items():
    try:
        cas_number = get_cas_from_text(ingredient)
    except AttributeError:
        cas_number = ""
    clean_name_str = clean_name(ingredient)
    empty_dict.update({
        'cas_number': cas_number,
        'clean_name': clean_name_str,
    })


In [42]:
cas_data_to_write = []
for ingredient, data in cas_metadata.items():
    cas_data_to_write.append([
        ingredient,
        data['cas_number'],
        data['clean_name']
    ])
import csv
with open('data/ingredients_cleaned.csv', 'w') as outfile:
    writer = csv.writer(outfile)
    writer.writerow(['original_ingredient', 'cas_number', 'clean_name'])
    writer.writerows(cas_data_to_write)

In [ ]:
#test by CAS number
test_cas = "13463-67-7"

url = f"{BASE_URL}/detail"
headers = {"X-Api-Key": API_KEY}
params = {"cas_rn": test_cas}

response = requests.get(url, headers=headers, params=params)

print(f"Status Code: {response.status_code}")
print(f"\nResponse JSON:")
print(response.json())

Status Code: 429

Response JSON:
{'message': 'Limit Exceeded'}


In [ ]:
#test by Name
test_name = "caffeine"

url = f"{BASE_URL}/search"
headers = {"X-Api-Key": API_KEY}
params = {"q": test_name}

response = requests.get(url, headers=headers, params=params)

print(f"Status Code: {response.status_code}")
print(f"\nResponse JSON:")
print(response.json())

In [ ]:
# API call
counter = 5
for ingredient, data in cas_metadata.items():

    cas_num = metadata['cas_number']
    clean_name_str = metadata['clean_name']

    if cas_num and cas_num != "":
        if isinstance(cas_num, list):
            cas_num = cas_num[0]

        url = f"{BASE_URL}/detail"
        headers = {"X-Api-Key": API_KEY}
        params = {"cas_rn": cas_num}
        response = requests.get(url, headers=headers, params=params)  

         if response.status_code == 200:
            official_name = response.json().get('name', '')
            metadata['official_name'] = official_name
            metadata['lookup_type'] = 'CAS'
        else:
            metadata['official_name'] = 'NOT FOUND'
            metadata['lookup_type'] = 'CAS' 
        
    elif clean_name_str and clean_name_str != "":
        url = f"{BASE_URL}/search"
        headers = {"X-Api-Key": API_KEY}
        params = {"q": clean_name_str}
        response = requests.get(url, headers=headers, params=params)

        if response.status_code == 200:
            results = response.json()
            if results.get('count', 0) > 0:
                official_name = results['results'][0].get('name', '')
                metadata['official_name'] = official_name
                metadata['lookup_type'] = 'NAME'
            else:
                metadata['official_name'] = 'NOT FOUND'
                metadata['lookup_type'] = 'NAME'
        else:
            metadata['official_name'] = 'NOT FOUND'
            metadata['lookup_type'] = 'NAME'
    
        else:
        metadata['official_name'] = 'NO DATA'
        metadata['lookup_type'] = 'NONE'

        print(f"Done: {ingredient[:30]}")



SyntaxError: illegal target for annotation (3948023891.py, line 5)

In [ ]:
final_data = []
for ingredient, metadata in cas_metadata.items():
    final_data.append([
        ingredient,
        metadata['cas_number'],
        metadata['clean_name'],
        metadata.get('official_name', ''),
        metadata.get('lookup_type', '')
    ])

with open('data/ingredients_final.csv', 'w', newline='') as outfile:
    writer = csv.writer(outfile)
    writer.writerow(['original_ingredient', 'cas_number', 'clean_name', 'official_name', 'lookup_type'])
    writer.writerows(final_data)


# Step Extract CAS numbers and clean names
print("Step 1: Processing ingredient names...")
unique_cas['CAS_Number'] = unique_cas['Ingredient Name'].apply(get_cas_from_text)
unique_cas['Clean_Name'] = unique_cas['Ingredient Name'].apply(clean_name)

# Step Save results
output_file = 'data/CDPH_ingredients_standardized.csv'
unique_cas.to_csv(output_file, index=False)

print(f"\n✓ Done! Results saved to: {output_file}")
print(f"\nSummary:")
print(f"Total rows: {len(unique_cas)}")
print(f"Rows with CAS numbers: {(unique_cas['CAS_Number'] != '').sum()}")
print(f"Official names retrieved: {(unique_cas['Official_Name'] != '').sum()}")

# Step 2: Get official names from API (only for rows that have CAS numbers)
print("\nStep 2: Getting official names from API...")
data['Official_Name'] = ""

rows_with_cas = data[data['CAS_Number'] != '']
total = len(rows_with_cas)

for i, (index, row) in enumerate(rows_with_cas.iterrows(), 1):
    print(f"[{i}/{total}] Getting name for CAS: {row['CAS_Number']}")
    official_name = get_official_name(row['CAS_Number'])
    data.at[index, 'Official_Name'] = official_name
    time.sleep(0.5)  # Be nice to the API